# Lentils foreign-object segmentation with DynUNet — inference

Loads a trained artifact (the [`01_train.ipynb`](01_train.ipynb) output by default; point
`LENTILS_PIPELINE_DIR` at a `train.py` run to evaluate a champion reproduction instead),
restores it with `CuvisPipeline.load_pipeline`, and

1. evaluates the test split — pixel **fg-IoU / fg-Dice** on object frames plus **image-level
   AUROC** (max foreground probability vs the frame-has-objects label) over all frames, and
2. renders per-frame overlays: false-color RGB, foreground probability, prediction vs GT.

At inference the graph runs only `Norm → DynUNet` (the losses are train/val/test-gated and the
augmentation is an identity passthrough); DynUNet tiles each full frame with Gaussian-blended
overlaps. `tile_batch=16` packs tiles onto the batch axis (~16× faster on the 2D mode,
output-identical).

In [ ]:
import utils

cfg = utils.resolve_config()
yaml_path, pt_path = utils.resolve_pipeline()
print("artifact:", yaml_path)

eng = utils.import_engine()
registry = eng.register_plugins(cfg["unet_manifest"], cfg["augment_manifest"])
pipe = eng.load_artifact(yaml_path, pt_path, registry=registry)
print("restored nodes:", [n.name for n in pipe.nodes])

## Evaluate

The tutorial artifact was trained on a few frames for a few epochs, so its numbers are far
below the champion reference — the comparison line shows the gap. For a full evaluation set
`MAX_FRAMES = 0` (all 180 test frames) and evaluate a champion-scale artifact.

In [ ]:
MAX_FRAMES = 8  # 0 = the full test split

splits_csv = (
    utils.LENTILS_NPZ_OUT / "lentils_seg_splits_fromhf.csv"
    if cfg["data_source"] == "hf"
    else cfg["splits_csv"]
)
metrics = eng.evaluate(pipe, splits_csv, split="test", max_frames=MAX_FRAMES)
champ = utils.CHAMPION["2d_128"]
print(
    f"frames: {metrics['frames']} ({metrics['object_frames']} object / "
    f"{metrics['normal_frames']} normal)\n"
    f"fg-IoU  = {metrics['fg_iou']:.4f}   (champion {champ['fg_iou']:.4f})\n"
    f"fg-Dice = {metrics['fg_dice']:.4f}   (champion {champ['fg_dice']:.4f})\n"
    f"image-AUROC (max-prob) = {metrics['image_auroc_maxprob']:.4f}   "
    f"(champion {champ['image_auroc']:.3f})"
)

## Visualize predictions

The two-node inference path, spelled out: z-score the full frame, tile it through DynUNet,
softmax the logits into a foreground-probability map, and argmax into a predicted mask.

In [ ]:
from pathlib import Path

import torch
from cuvis_ai_schemas.enums import ExecutionStage
from cuvis_ai_schemas.execution import Context


def predict_frame(pipeline, npz_path):
    """Full-frame tiled inference -> (fg_prob [H,W], pred_mask [H,W])."""
    nodes = {n.name: n for n in pipeline.nodes}
    norm, net = nodes["Norm"], nodes["DynUNet"]
    device = next(iter(net.parameters())).device
    frame = utils.load_lentils_frame(npz_path)
    cube = torch.from_numpy(frame["cube"]).unsqueeze(0).to(device)
    with torch.no_grad():
        normed = norm.forward(data=cube)["normalized"]
        ctx = Context(stage=ExecutionStage.INFERENCE, batch_idx=0, global_step=0)
        logits = net.forward(normed, context=ctx)["logits"][0]  # [H, W, K]
        fg_prob = torch.softmax(logits.float(), dim=-1)[..., 1].cpu().numpy()
        pred = (logits.argmax(-1) >= 1).cpu().numpy()
    return frame, fg_prob, pred


# pick the first object frames of the test split
test_frames = eng.split_frames(splits_csv, "test")
object_frames = [p for p in test_frames if utils.load_lentils_frame(p)["mask"].any()][:3]
for p in object_frames:
    frame, fg_prob, pred = predict_frame(pipe, p)
    utils.render_segmentation_panel(
        frame["cube"], fg_prob, pred,
        wavelengths=frame["wavelengths"], gt_mask=frame["mask"],
        title=Path(p).name,
    )

## Going further

- `examples/lentils/profile_pipeline.py` sweeps `tile_overlap` × `tile_batch` with cuvis-ai's
  built-in per-node profiler.
- `tile_overlap=0` (with `tile_gaussian=false`) is a ~3.3× faster eval mode at ~1.5–2 %
  relative fg-IoU cost; with `tile_batch=16` the full-quality `0.5` overlap already runs at
  ~21 fps on an RTX 4090, so the trade-off is rarely needed.
- Data loading (~1 s per 263 MB compressed frame) dominates end-to-end throughput — see the
  known-limitations section of the README.